# NT120 — Lecture 03 code notes

Các ví dụ Python/NumPy/Pandas được chép và sắp xếp lại từ *Lecture 03: Python, Google Colab and Tabular Data*. Notebook dùng một bảng dữ liệu nhỏ mô phỏng để toàn bộ phần thực hành có thể chạy từ trên xuống. Xem `README.md` để cài Miniconda và cấu hình môi trường.

In [1]:
# Slide 37 — Standard header
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 40)

# Works whether the notebook is opened from the project root or notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)
FLOW_PATH = DATA_DIR / "flows.csv"
print("pandas:", pd.__version__, "| numpy:", np.__version__)

pandas: 2.3.3 | numpy: 2.2.6


## Colab và thứ tự chạy cell

Slide 9 dùng `!` (shell) và `%` (notebook magic). Các lệnh này chạy trực tiếp trên Google Colab; phần Drive chỉ bật khi notebook đang chạy trong Colab.

In [2]:
# Slide 9 — Colab idioms (bỏ comment khi chạy trên Colab)
# !pip list | grep -i pandas
# !ls -lh /content
# %timeit np.arange(1_000_000).sum()
# from google.colab import files
# uploaded = files.upload()
# from google.colab import drive
# drive.mount("/content/drive")
# df.to_csv("/content/drive/MyDrive/nt120/out.csv", index=False)

# Slide 10 — kernel giữ biến giữa các cell
t = 5
n = sum(s > t for s in [1, 3, 8, 10])
print(t, n)

5 2


In [3]:
# Dữ liệu nhỏ mô phỏng cho các ví dụ Pandas bên dưới
raw = pd.DataFrame({
    "dst_port": [22, 80, 443, 8080, 53, 22, 80, 3389],
    "duration": [3.395, 61.5, 0.04, 120.0, 0.0, 4.1, 0.9, 0.02],
    "fwd_pkts": [8, 12, 1, 20, 2, 6, 3, 1],
    "bwd_pkts": [7, 10, 0, 19, 1, 4, 2, 0],
    "fwd_bytes": [6520, 22000, 0, 402110, 95, 7047, 250, 60],
    "bwd_bytes": ["6,520", "22,000", "0", "402,110", "95", "7,047", "250", "60"],
    "protocol": ["TCP", "UDP", "TCP", "TCP", "UDP", "TCP", "UDP", "TCP"],
    "label": ["BENIGN", "BENIGN", "PortScan", "BENIGN", "BENIGN", "BENIGN", "BENIGN", "PortScan"],
    "ttl": [64, "unknown", 128, 64, 255, 64, 64, 128],
    "start": pd.date_range("2026-09-14", periods=8, freq="D"),
})
raw.to_csv(FLOW_PATH, index=False)
raw.head()

,dst_port,duration,fwd_pkts,bwd_pkts,fwd_bytes,bwd_bytes,protocol,label,ttl,start
0,22,3.395,8,7,6520,"6,520",TCP,BENIGN,64,2026-09-14
1,80,61.500,12,10,22000,"22,000",UDP,BENIGN,unknown,2026-09-15
2,443,0.040,1,0,0,0,TCP,PortScan,128,2026-09-16
3,8080,120.000,20,19,402110,"402,110",TCP,BENIGN,64,2026-09-17
4,53,0.000,2,1,95,95,UDP,BENIGN,255,2026-09-18


In [4]:
# Slide 22 — validation function
def validate(df):
    text = set(df.columns) - set(df.select_dtypes("number").columns)
    assert text == {"protocol", "label", "start", "ttl"}, (
        f"unexpected text columns: {sorted(text)}"
    )
    assert set(df["label"]) <= {"BENIGN", "PortScan"}
    assert df["dst_port"].between(0, 65535).all()
    assert (df["duration"] >= 0).all()
    assert not df.duplicated().any()
    return df

# Bản slide gốc dùng schema chỉ gồm protocol và label là cột text.
# Dữ liệu mô phỏng thêm start/ttl để minh hoạ các slide sau.

## Python cơ bản — slides 33 và 35

In [5]:
# Slide 33 — lists, dicts, comprehensions
ports = [22, 80, 443, 3389]
names = {22: "ssh", 80: "http", 443: "https"}
known = [p for p in ports if p in names]
labels = [names.get(p, "other") for p in ports]

def rate(bytes_, seconds):
    if seconds <= 0:
        return 0.0
    return bytes_ / seconds

known, labels, rate(6520, 3.395)

([22, 80, 443], ['ssh', 'http', 'https', 'other'], 1920.4712812960236)

In [6]:
# Slide 35 — loops, comprehension, function, f-string
sizes = [527, 6520, 268, 3728, 11409]
big = []
for b in sizes:
    if b > 1000:
        big.append(b / 1000)

big = [b / 1000 for b in sizes if b > 1000]
total = sum(sizes)
share = {i: b / total for i, b in enumerate(sizes)}

def mbps(nbytes, seconds):
    """Megabits per second, 0.0 if seconds <= 0."""
    if seconds <= 0:
        return 0.0
    return 8 * nbytes / 1e6 / seconds

print(f"{mbps(6_520, 3.395):.3f} Mbit/s")

0.015 Mbit/s


## NumPy — slides 40, 42 và 43

In [7]:
# Slide 40 — creating and inspecting arrays
a = np.array([[4.12, 9, 1842, 0.32],
              [0.03, 2, 0, 0.00],
              [61.5, 318, 402110, 0.71]])
print(a.shape, a.ndim, a.dtype)
print(a[0, 2])
print(a[:, 1])
print(a[a[:, 0] > 1])
print(a.astype("int64"))
print(np.arange(0, 10, 2), np.linspace(0, 1, 5), np.zeros((2, 3)), sep="\n")

(3, 4) 2 float64
1842.0
[  9.   2. 318.]
[[4.1200e+00 9.0000e+00 1.8420e+03 3.2000e-01]
 [6.1500e+01 3.1800e+02 4.0211e+05 7.1000e-01]]
[[     4      9   1842      0]
 [     0      2      0      0]
 [    61    318 402110      0]]
[0 2 4 6 8]
[0.   0.25 0.5  0.75 1.  ]
[[0. 0. 0.]
 [0. 0. 0.]]


In [8]:
# Slides 42–43 — vectorization, masks, missing values
durations = np.array([3.4, 0.0, 61.5, 0.02])
byte_counts = np.array([7047, 95, 22600, 60])
rates = np.where(durations > 0, byte_counts / durations, 0.0)

d = durations
b = byte_counts
mask = d > 0
print(mask, mask.sum(), mask.mean(), b[mask])
print(np.where(mask, b / d, 0.0))
print(np.mean([1.0, np.nan]), np.nanmean([1.0, np.nan]), np.isnan(np.nan))

[ True False  True  True] 3 0.75 [ 7047 22600    60]
[2072.64705882    0.          367.4796748  3000.        ]
nan 1.0 True


/tmp/ipykernel_27240/3840650423.py:4: RuntimeWarning: divide by zero encountered in divide
  rates = np.where(durations > 0, byte_counts / durations, 0.0)
/tmp/ipykernel_27240/3840650423.py:10: RuntimeWarning: divide by zero encountered in divide
  print(np.where(mask, b / d, 0.0))


## Đọc, kiểm tra, chọn và nhóm dữ liệu — slides 47, 49, 54, 55 và 57

In [9]:
# Slide 47 — read_csv variations
df = pd.read_csv(FLOW_PATH, thousands=",")
head = pd.read_csv(FLOW_PATH, nrows=1000)
subset_at_load = pd.read_csv(
    FLOW_PATH,
    usecols=["dst_port", "duration", "label"],
    dtype={"dst_port": "int32"},
    na_values=["unknown", "?"],
)
df.dtypes

dst_port       int64
duration     float64
fwd_pkts       int64
bwd_pkts       int64
fwd_bytes      int64
bwd_bytes      int64
protocol      object
label         object
ttl           object
start         object
dtype: object

In [10]:
# Slide 49 — six lines for a new CSV
print("shape:", df.shape)
df.info()
display(df.describe())
display(df["label"].value_counts(normalize=True))
display(df.isna().sum().sort_values(ascending=False).head(10))
df.head(3)

shape: (8, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 10 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   dst_port   8 non-null      int64  
 1   duration   8 non-null      float64
 2   fwd_pkts   8 non-null      int64  
 3   bwd_pkts   8 non-null      int64  
 4   fwd_bytes  8 non-null      int64  
 5   bwd_bytes  8 non-null      int64  
 6   protocol   8 non-null      object 
 7   label      8 non-null      object 
 8   ttl        8 non-null      object 
 9   start      8 non-null      object 
dtypes: float64(1), int64(5), object(4)
memory usage: 768.0+ bytes


,dst_port,duration,fwd_pkts,bwd_pkts,fwd_bytes,bwd_bytes
count,8.000000,8.000000,8.000000,8.000000,8.000000,8.000000
mean,1521.125000,23.744375,6.625000,5.375000,54760.250000,54760.250000
std,2890.229276,44.241279,6.631903,6.545173,140549.542863,140549.542863
min,22.000000,0.000000,1.000000,0.000000,0.000000,0.000000
25%,45.250000,0.035000,1.750000,0.750000,86.250000,86.250000
50%,80.000000,2.147500,4.500000,3.000000,3385.000000,3385.000000
75%,1179.500000,18.450000,9.000000,7.750000,10785.250000,10785.250000
max,8080.000000,120.000000,20.000000,19.000000,402110.000000,402110.000000


label
BENIGN      0.75
PortScan    0.25
Name: proportion, dtype: float64

dst_port     0
duration     0
fwd_pkts     0
bwd_pkts     0
fwd_bytes    0
bwd_bytes    0
protocol     0
label        0
ttl          0
start        0
dtype: int64

,dst_port,duration,fwd_pkts,bwd_pkts,fwd_bytes,bwd_bytes,protocol,label,ttl,start
0,22,3.395,8,7,6520,6520,TCP,BENIGN,64,2026-09-14
1,80,61.500,12,10,22000,22000,UDP,BENIGN,unknown,2026-09-15
2,443,0.040,1,0,0,0,TCP,PortScan,128,2026-09-16


In [11]:
# Slides 54–55 — selection and boolean masks
dur = df["duration"]
sub = df[["duration", "fwd_pkts", "label"]]
attacks = df[df["label"] != "BENIGN"]
long_udp = df[(df["protocol"] == "UDP") & (df["duration"] > 60)]
web_ports = df[df["dst_port"].isin([80, 443, 8080])]
m1 = df["duration"] > 1
m2 = df["protocol"] == "UDP"
display(df.loc[df["label"] == "PortScan", ["duration", "fwd_pkts"]])
display(df[m1 & m2])
print("both:", (m1 & m2).sum(), "| either:", (m1 | m2).sum(), "| not m1:", (~m1).sum())

,duration,fwd_pkts
2,0.04,1
7,0.02,1


,dst_port,duration,fwd_pkts,bwd_pkts,fwd_bytes,bwd_bytes,protocol,label,ttl,start
1,80,61.5,12,10,22000,22000,UDP,BENIGN,unknown,2026-09-15


both: 1 | either: 6 | not m1: 4


In [12]:
# Slide 57 — groupby and crosstab
cols = ["duration", "fwd_pkts", "fwd_bytes"]
display(df.groupby("label")[cols].agg(["mean", "median"]).round(2))
pd.crosstab(df["protocol"], df["label"])

duration        fwd_pkts        fwd_bytes        
             mean median     mean median      mean  median
label                                                     
BENIGN      31.65   3.75      8.5    7.0  73003.67  6783.5
PortScan     0.03   0.03      1.0    1.0     30.00    30.0

label,BENIGN,PortScan
protocol,,
TCP,3,2
UDP,3,0


## Join, cột dẫn xuất và làm sạch — slides 59, 60, 63–66

In [13]:
# Slide 59 — left join with key validation
services = pd.DataFrame({"dst_port": [22, 80, 443], "service": ["ssh", "http", "https"]})
n = len(df)
m = df.merge(services, on="dst_port", how="left", validate="many_to_one")
assert len(m) == n
m["service"] = m["service"].fillna("other")

bad = pd.DataFrame({"dst_port": [22, 80, 80, 443], "service": ["ssh", "http", "web", "https"]})
print("rows without validation:", len(df.merge(bad, on="dst_port", how="left")))
try:
    df.merge(bad, on="dst_port", how="left", validate="many_to_one")
except pd.errors.MergeError as error:
    print(type(error).__name__, "— duplicate lookup key detected")
m.head()

rows without validation: 10
MergeError — duplicate lookup key detected


,dst_port,duration,fwd_pkts,bwd_pkts,fwd_bytes,bwd_bytes,protocol,label,ttl,start,service
0,22,3.395,8,7,6520,6520,TCP,BENIGN,64,2026-09-14,ssh
1,80,61.500,12,10,22000,22000,UDP,BENIGN,unknown,2026-09-15,http
2,443,0.040,1,0,0,0,TCP,PortScan,128,2026-09-16,https
3,8080,120.000,20,19,402110,402110,TCP,BENIGN,64,2026-09-17,other
4,53,0.000,2,1,95,95,UDP,BENIGN,255,2026-09-18,other


In [14]:
# Slide 60 — derived columns
df = df.assign(
    bytes_total=df["fwd_bytes"] + df["bwd_bytes"],
    bytes_per_pkt=lambda d: d["bytes_total"] / (d["fwd_pkts"] + d["bwd_pkts"]),
    is_web=df["dst_port"].isin([80, 443, 8080]),
    dur_class=pd.cut(df["duration"], [-0.001, 0.1, 10, 121], labels=["short", "medium", "long"]),
)
display(df["dur_class"].value_counts())
display(df.sort_values("duration", ascending=False).head(3))
df.nlargest(3, "fwd_bytes")

dur_class
short     3
medium    3
long      2
Name: count, dtype: int64

,dst_port,duration,fwd_pkts,bwd_pkts,fwd_bytes,bwd_bytes,protocol,label,ttl,start,bytes_total,bytes_per_pkt,is_web,dur_class
3,8080,120.0,20,19,402110,402110,TCP,BENIGN,64,2026-09-17,804220,20621.025641,True,long
1,80,61.5,12,10,22000,22000,UDP,BENIGN,unknown,2026-09-15,44000,2000.000000,True,long
5,22,4.1,6,4,7047,7047,TCP,BENIGN,64,2026-09-19,14094,1409.400000,False,medium


,dst_port,duration,fwd_pkts,bwd_pkts,fwd_bytes,bwd_bytes,protocol,label,ttl,start,bytes_total,bytes_per_pkt,is_web,dur_class
3,8080,120.0,20,19,402110,402110,TCP,BENIGN,64,2026-09-17,804220,20621.025641,True,long
1,80,61.5,12,10,22000,22000,UDP,BENIGN,unknown,2026-09-15,44000,2000.000000,True,long
5,22,4.1,6,4,7047,7047,TCP,BENIGN,64,2026-09-19,14094,1409.400000,False,medium


In [15]:
# Slide 63 — convert text safely
small = raw[["ttl"]].copy()
before = small["ttl"].isna().sum()
small["ttl"] = pd.to_numeric(small["ttl"], errors="coerce")
lost = small["ttl"].isna().sum() - before
print(lost, "values could not be parsed")

df["bwd_bytes"] = pd.to_numeric(df["bwd_bytes"], errors="raise")
print(df["bwd_bytes"].dtype)
df["bwd_bytes"].describe()

1 values could not be parsed
int64


count         8.000000
mean      54760.250000
std      140549.542863
min           0.000000
25%          86.250000
50%        3385.000000
75%       10785.250000
max      402110.000000
Name: bwd_bytes, dtype: float64

In [16]:
# Slides 64–65 — duplicates, outliers, encoding, scaling
print("duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()
print("negative durations:", (df["duration"] < 0).sum())
assert df["dst_port"].between(0, 65535).all()

train = df.sample(frac=0.75, random_state=RANDOM_STATE)
mu, sd = train["duration"].mean(), train["duration"].std()
outlier = (df["duration"] - mu).abs() > 3 * sd
print("outlier share:", outlier.mean())

display(pd.get_dummies(df["protocol"], prefix="proto"))
display(pd.cut(df["dst_port"], [-1, 1023, 49151, 65535], labels=["well-known", "registered", "dynamic"]))
z = (df["duration"] - mu) / sd
vocab = sorted(train["protocol"].unique())
print("vocabulary from training:", vocab)

duplicate rows: 0
negative durations: 0
outlier share: 0.125


,proto_TCP,proto_UDP
0,True,False
1,False,True
2,True,False
3,True,False
4,False,True
5,True,False
6,False,True
7,True,False


0    well-known
1    well-known
2    well-known
3    registered
4    well-known
5    well-known
6    well-known
7    registered
Name: dst_port, dtype: category
Categories (3, object): ['well-known' < 'registered' < 'dynamic']

vocabulary from training: ['TCP', 'UDP']


In [17]:
# Slide 66 — raw in, clean and validated out
def clean(raw):
    df = raw.copy()
    df["bwd_bytes"] = pd.to_numeric(
        df["bwd_bytes"].astype(str).str.replace(",", "")
    )
    return df.drop_duplicates()

raw = pd.read_csv(FLOW_PATH)
df_clean = validate(clean(raw))
df_clean.dtypes

dst_port       int64
duration     float64
fwd_pkts       int64
bwd_pkts       int64
fwd_bytes      int64
bwd_bytes      int64
protocol      object
label         object
ttl           object
start         object
dtype: object

## Tái lập kết quả và checkpoint — slides 70, 71 và 75

In [18]:
# Slide 70 — seeded randomness and a fixed reference date
rng = np.random.default_rng(42)
print(rng.normal(size=3))
print(df_clean.sample(frac=0.1, random_state=42))
df_clean["start"] = pd.to_datetime(df_clean["start"], errors="raise")
REF = pd.Timestamp("2026-09-21")
df_clean["age_days"] = (REF - df_clean["start"]).dt.days
df_clean[["start", "age_days"]].head()

[ 0.30471708 -1.03998411  0.7504512 ]
   dst_port  duration  fwd_pkts  bwd_pkts  fwd_bytes  bwd_bytes protocol  \
1        80      61.5        12        10      22000      22000      UDP   

    label      ttl       start  
1  BENIGN  unknown  2026-09-15  


TypeError: unsupported operand type(s) for -: 'Timestamp' and 'str'

In [ ]:
# Slide 71 — an idempotent transformation
# Unsafe toggle shown in the slide (do not run twice):
# df["duration"] = df["duration"] / 60

# Safe switch: derive from raw values; rerunning returns the same answer.
df_clean["duration_min"] = raw["duration"] / 60
df_clean["bwd_bytes"] = pd.to_numeric(
    df_clean["bwd_bytes"].astype(str).str.replace(",", "")
)
df_clean[["duration", "duration_min", "bwd_bytes"]].head()

In [ ]:
# Slide 75 — checkpoint
checkpoint = pd.DataFrame({"port": [22, 80, 22, 443], "bytes": [100, 250, 90, 500]})
print(checkpoint.shape)
print(checkpoint[checkpoint["port"] == 22]["bytes"].mean())
print(checkpoint.groupby("port")["bytes"].sum().loc[22])

## Appendix — đầy đủ lệnh tham khảo (slides 82–85)

Các cell dưới đây bổ sung những lệnh ngắn trong phần Appendix của slide. Chúng dùng dữ liệu mô phỏng đã tạo ở trên để có thể chạy thử.

In [ ]:
# Slide 82 — Pandas: inspect and select
ref = df_clean.copy()
print(ref.shape, len(ref))
ref.info()
display(ref.dtypes)
display(ref.describe())
display(ref["duration"].describe())
display(ref.head())
display(ref.sample(3, random_state=42))
display(ref["label"].value_counts(normalize=True))
display(ref.isna().sum())
display(ref.isna().mean())
print(ref.duplicated().sum(), ref["protocol"].nunique(), ref["protocol"].unique())
display(ref["duration"])
display(ref[["duration", "fwd_pkts"]])
display(ref[ref["duration"] > 0])
display(ref.query("duration > 0"))
display(ref[ref["dst_port"].isin([22, 80])])
display(ref.loc[ref["duration"] > 0, ["duration", "protocol"]])
ref.iloc[0:5, 0:3]

In [ ]:
# Slide 83 — Pandas: transform, group, combine and save
s = pd.Series(["6,520", "unknown", "  TCP "], name="example")
numeric_s = pd.to_numeric(s.str.replace(",", ""), errors="coerce")
clean_text = s.str.replace(",", "").str.lower().str.strip()
display(numeric_s)
display(clean_text)
display(ref.fillna(0))
display(ref.dropna(subset=["duration"]))
transformed = ref.assign(
    rate=np.where(ref["duration"] > 0, ref["fwd_bytes"] / ref["duration"], 0.0),
    duration_band=pd.cut(ref["duration"], bins=3),
    duration_quartile=pd.qcut(ref["duration"], 4, duplicates="drop"),
)
display(transformed)
display(ref.groupby("protocol")["duration"].agg(["mean", "median", "size"]))
display(pd.crosstab(ref["protocol"], ref["label"]))
display(ref.pivot_table(index="protocol", columns="label", values="duration", aggfunc="mean"))
lookup = pd.DataFrame({"dst_port": [22, 80, 443], "name": ["ssh", "http", "https"]})
display(ref.merge(lookup, on="dst_port", how="left", validate="many_to_one"))
display(ref.sort_values("duration", ascending=False))
display(ref.nlargest(3, "fwd_bytes"))
# ref.to_csv("out.csv", index=False)
# ref.to_parquet("out.parquet")  # requires pyarrow or fastparquet

In [ ]:
# Slide 84 — NumPy reference
na = np.array([[1.0, 2.0, np.nan], [4.0, 5.0, 6.0]])
print(np.zeros((2, 3)))
print(np.arange(0, 10, 2), np.linspace(0, 1, 5))
print(na.shape, na.ndim, na.dtype, na.size)
print(na.reshape(3, 2), na.T, na.astype("float64"), sep="\n")
print(na[0, 2], na[:, 1], na[na[:, 0] > 1])
print(np.nansum(na, axis=0), np.nanmean(na, axis=1), np.nanstd(na, axis=0))
numpy_mask = na[:, 0] > 1
print(np.where(numpy_mask, "large", "small"), numpy_mask.sum(), np.any(numpy_mask), np.all(numpy_mask))
print(np.isnan(na), np.nanmean(na), np.nan_to_num(na))
print(np.log10(np.array([1.0, 10.0, 100.0])), np.log1p(np.array([0.0, 1.0, 9.0])))
rng = np.random.default_rng(42)
rng.normal(size=3)

In [ ]:
# Slide 85 — Colab troubleshooting commands (run in a Colab code cell)
# !ls
# !pip install package_name
# After installing a package, use Runtime > Restart session if Colab asks you to.